# HW2 --- Data-Based (Surrogate) Modeling

Fill in the `# TODO` sections below to answer each question in `HW2.pdf`.
Dataset: [`aero_naca4_airfoil_panel.csv`](https://github.com/Ahmed-Bayoumy/MECH559/blob/DEV/Datasets/aero_naca4_airfoil_panel.csv)
(NACA 4-digit airfoil, vortex-panel method, $p=500$).

- **Inputs (6):** `max_camber_m`, `camber_pos_p`, `thickness_t`, `alpha_deg`, `mach`, `reynolds`
  (use $\log_{10}$(reynolds) in place of the raw column).
- **Output:** `CL`.
- **"Scaled inputs"** = per-column min--max scaling to $[0,1]$ using **training-set** bounds.

Starter code from Lecture 4's notebooks is linked in `HW2.pdf`; adapt it rather than
writing every model from a blank page. Each question below is copied verbatim from `HW2.pdf`;
Rename this file `<McGill_ID>_HW2.ipynb` before submitting.

## Setup: imports and data

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF as RBFKernel, ConstantKernel
from sklearn.model_selection import KFold
from scipy.interpolate import RBFInterpolator
from scipy.spatial.distance import cdist

RNG = 559  # fixed random seed -- pass it as random_state / default_rng(RNG) wherever a seed is needed

CSV_URL = "https://raw.githubusercontent.com/Ahmed-Bayoumy/MECH559/DEV/Datasets/aero_naca4_airfoil_panel.csv"
df = pd.read_csv(CSV_URL)
train = df[df.split == 'train'].reset_index(drop=True)
test = df[df.split == 'test'].reset_index(drop=True)
print(f"total={len(df)} train={len(train)} test={len(test)}")

raw_cols = ['max_camber_m', 'camber_pos_p', 'thickness_t', 'alpha_deg', 'mach']
feat_names = raw_cols + ['log10_reynolds']

def make_raw(d):
    """Return the 6 raw (unscaled) input columns, with reynolds -> log10(reynolds)."""
    X = d[raw_cols].to_numpy(dtype=float).copy()
    log10re = np.log10(d['reynolds'].to_numpy(dtype=float))
    return np.column_stack([X, log10re])

Xtr = make_raw(train); ytr = train['CL'].to_numpy(dtype=float)
Xte = make_raw(test);  yte = test['CL'].to_numpy(dtype=float)

def rmse(y, yhat):
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def r2(y, yhat):
    ss_res = np.sum((y - yhat) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    return float(1 - ss_res / ss_tot)

# One set of 5 CV folds, created once and reused in Q3, Q4 and Q5, so every
# model and every hyperparameter value is scored on the same folds.
kf = KFold(n_splits=5, shuffle=True, random_state=RNG)

print("Xtr shape:", Xtr.shape, " Xte shape:", Xte.shape)

## Question 1 --- Design of experiments

[`aero_naca4_airfoil_panel.csv`](https://github.com/Ahmed-Bayoumy/MECH559/blob/DEV/Datasets/aero_naca4_airfoil_panel.csv) was sampled with LHS ($n=6$, $p=500$), not a full-factorial grid. An LHS with $p$ points has $p$ distinct levels in every 1-D projection. Compute the full-factorial grid size needed to match this 1-D resolution ($p^{n}$ runs) and compare it to $p$; conversely, find how many levels per dimension a grid of at most $p$ runs can afford ($\lfloor p^{1/n}\rfloor$). Briefly explain why LHS avoids this *curse of dimensionality* while still space-filling well in 1-D.

In [ ]:
n, p = 6, 500

# TODO: compute the full-factorial grid size needed to match p's 1-D resolution
# hint: grid_full = p ** n

# TODO: compute how many levels/dimension a budget of p runs can afford
# hint: levels_afford = int(np.floor(p ** (1 / n)))

# TODO: print both results and compare grid_full to p

## Question 2 --- Polynomial regression from first principles

Using the `train` rows, fit an affine model $\hat{y}(\mathbf{x};\mathbf{w}) = w_0 + \sum_{j=1}^{6} w_j x_j$ for `CL` (inputs: `max_camber_m`, `camber_pos_p`, `thickness_t`, `alpha_deg`, `mach`, $\log_{10}$(`reynolds`)) by solving the normal equations $\mathbf{Z}^{\mathsf T}\mathbf{Z}\,\mathbf{w} = \mathbf{Z}^{\mathsf T}\mathbf{y}$ directly -- adapt the Starter Code notebook's `normal_equations_fit`, or assemble $\mathbf{Z}$ yourself and pass $\mathbf{Z}^{\mathsf T}\mathbf{Z}$ and $\mathbf{Z}^{\mathsf T}\mathbf{y}$ to `numpy.linalg.solve` or MATLAB's backslash operator (no canned regression function; note that `lstsq`, or backslash applied to $\mathbf{Z}$ and $\mathbf{y}$ directly, does *not* form $\mathbf{Z}^{\mathsf T}\mathbf{Z}$). Report $\mathbf{w}$ and the test RMSE/$R^2$, then confirm both match `scikit-learn`'s `LinearRegression` (or MATLAB's `fitlm`).

In [ ]:
# TODO: assemble the design matrix Z = [1 | Xtr] (intercept column + 6 raw inputs)
# hint: Ztr = np.column_stack([np.ones(len(Xtr)), Xtr])
# hint: Zte = np.column_stack([np.ones(len(Xte)), Xte])

# TODO: solve the normal equations directly -- do NOT call a regression library here
# hint: w = np.linalg.solve(Ztr.T @ Ztr, Ztr.T @ ytr)

# TODO: predict on the test set and report RMSE/R^2
# hint: yhat_te = Zte @ w
# hint: print(rmse(yte, yhat_te), r2(yte, yhat_te))

# TODO: refit with scikit-learn's LinearRegression and confirm it matches
# hint: lr = LinearRegression().fit(Xtr, ytr)
# hint: print(lr.intercept_, lr.coef_)

## Question 3 --- Ill-conditioning and ridge regularization

Extend question 2 to a pure quadratic (add $w_{j+6}\,x_j^2$, 13 columns in $\mathbf{Z}$) and report $\kappa(\mathbf{Z}^{\mathsf T}\mathbf{Z})$. Min-max scale the inputs to $[0,1]$ (training-set bounds), rebuild $\mathbf{Z}$, and report the new condition number. Sweep the ridge penalty $\lambda \in \{0, 10^{-4}, 10^{-2}, 1, 100\}$ (adapt `ridge_fit`, i.e. $\mathbf{w}_{\text{ridge}} = (\mathbf{Z}^{\mathsf T}\mathbf{Z}+\lambda\mathbf{I})^{-1}\mathbf{Z}^{\mathsf T}\mathbf{y}$ on the scaled $\mathbf{Z}$; this penalizes $w_0$ too, so if you use `sklearn.linear_model.Ridge` pass your own column of ones with `fit_intercept=False` to get the same result); using 5-fold CV on the training set, plot CV RMSE vs. $\lambda$, report the best $\lambda$, and compare its test RMSE/$R^2$ to $\lambda=0$. **Relate this to the bias--variance trade-off and to $\kappa(\mathbf{Z}^{\mathsf T}\mathbf{Z}+\lambda\mathbf{I})$.**

In [ ]:
# TODO: build the pure-quadratic design matrix (intercept + 6 linear + 6 squared terms, 13 cols)
# hint: def make_quad(Xraw): return np.column_stack([Xraw, Xraw**2])
# hint: Ztr_quad_raw = np.column_stack([np.ones(len(Xtr)), make_quad(Xtr)])

# TODO: report kappa(Z^T Z) on the UNSCALED quadratic features
# hint: np.linalg.cond(Ztr_quad_raw.T @ Ztr_quad_raw)

# TODO: min-max scale the 6 inputs to [0,1] using TRAINING bounds, then rebuild the quadratic Z
# hint: lo = Xtr.min(axis=0); hi = Xtr.max(axis=0)
# hint: def scale(X): return (X - lo) / (hi - lo)
# hint: Xtr_s, Xte_s = scale(Xtr), scale(Xte)

# TODO: report kappa(Z^T Z) on the SCALED quadratic features and compare to the unscaled value
# hint: Ztr_quad_s = np.column_stack([np.ones(len(Xtr_s)), make_quad(Xtr_s)])
# hint: Zte_quad_s = np.column_stack([np.ones(len(Xte_s)), make_quad(Xte_s)])

# TODO: sweep lambda in {0, 1e-4, 1e-2, 1, 100}; for each, run 5-fold CV on the training set
#       using the kf folds created in Setup (do NOT create new folds per lambda)
# hint: for tr_idx, va_idx in kf.split(Ztr_quad_s): ...
# hint: Z_f, y_f = Ztr_quad_s[tr_idx], ytr[tr_idx]
# hint: w_ridge = np.linalg.solve(Z_f.T @ Z_f + lam * np.eye(Z_f.shape[1]), Z_f.T @ y_f)

# TODO: plot mean CV RMSE vs. lambda (semilogx), pick the best lambda

# TODO: fit on the full training set at lambda=0 and at the best lambda; compare test RMSE/R^2

## Question 4 --- RBF and its spread parameter

Fit a Gaussian RBF surrogate $\phi(r)=\exp(-\lambda r^2)$ for `CL` (same 6 scaled inputs) with the Starter Code notebook (replace its 1-D distance by the Euclidean distance in 6-D) or a library, sweeping the spread over $\geq4$ values across orders of magnitude (e.g. $\lambda\in[10^{-4},10^{3}]$; large $\lambda$ = narrow basis functions). If you use `scipy.interpolate.RBFInterpolator`, set `kernel='gaussian'` and `epsilon` $=\sqrt{\lambda}$ -- the default thin-plate kernel ignores `epsilon`; MATLAB's `newrbe` spread $s$ corresponds to $\lambda=(0.8326/s)^2$. Using 5-fold CV, plot CV RMSE vs. spread and identify the best value. **Relate the too-narrow/too-wide extremes to the "good spread" vs. "poor spread" behavior from lecture.**

In [ ]:
# TODO: pick >= 4 spread values (lambda) spanning multiple orders of magnitude
# hint: spreads = [1e-3, 1e-2, 1e-1, 1, 10, 100]

# TODO: for each spread, run 5-fold CV (kf folds from Setup) using RBFInterpolator
# hint: eps = np.sqrt(lam)
# hint: rbfi = RBFInterpolator(Xtr_f, ytr_f, kernel='gaussian', epsilon=eps)
# hint: fold_rmse = rmse(yva_f, rbfi(Xva_f))

# TODO: plot mean CV RMSE vs. spread (loglog), identify the best spread

# TODO: fit on the full (scaled) training set at the best spread; report test RMSE/R^2

# TODO: comment on the too-narrow vs. too-wide extremes of your sweep

## Question 5 --- Neural network width as a complexity knob

Fit a single-hidden-layer network for `CL` (same 6 scaled inputs) with `MLPRegressor` (use `solver='lbfgs'`, `max_iter=5000`, a fixed `random_state`; the default `adam` with 200 iterations does not converge here) or `fitrnet`, sweeping the hidden-layer width over $\geq4$ values (e.g. 2, 8, 32, 64). Using 5-fold CV, plot CV RMSE vs. width and identify where under- and overfitting set in.

In [ ]:
# TODO: pick >= 4 widths, e.g. widths = [2, 8, 32, 64]

# TODO: for each width, run 5-fold CV (kf folds from Setup) using MLPRegressor
# hint: mlp = MLPRegressor(hidden_layer_sizes=(width,), solver='lbfgs',
#                           max_iter=5000, random_state=RNG)
# note: this keeps sklearn's defaults activation='relu' and alpha=1e-4 (an L2 penalty,
#       i.e. a second complexity knob); state this in your answer.
# hint: mlp.fit(Xtr_f, ytr_f); fold_rmse = rmse(yva_f, mlp.predict(Xva_f))

# TODO: plot mean CV RMSE vs. width (semilogx)

# TODO: fit on the full training set at the best width; report test RMSE/R^2

# TODO: comment on where the curve stops improving / starts overfitting

## Question 6 --- Kriging: correlation-matrix conditioning and uncertainty

Draw a fixed random subsample of $p=25$ training points (e.g. `np.random.default_rng(559)`) and use scaled inputs throughout this question. Using $R(\mathbf{x}_i,\mathbf{x}_j) = \exp(-\theta\lVert\mathbf{x}_i-\mathbf{x}_j\rVert^2)$ and the ordinary-Kriging equations from lecture ($\hat\beta = (\mathbf{1}^{\mathsf T}\mathbf{R}^{-1}\mathbf{1})^{-1}\mathbf{1}^{\mathsf T}\mathbf{R}^{-1}\mathbf{y}$, $\hat y(\mathbf{x}_{\text{new}}) = \hat\beta + \mathbf{r}^{\mathsf T}\mathbf{R}^{-1}(\mathbf{y}-\hat\beta\mathbf{1})$), adapt the Starter Code notebook's Kriging fit/predict (not a black-box library) for this sub-step. For $\geq3$ values of $\theta$ across orders of magnitude (e.g. $10^{-3}$ to $10^{2}$; call `kriging_fit` with `nugget=0`, since its default is $10^{-10}$), report $\kappa(\mathbf{R})$ and **comment on what a large value implies for computing $\mathbf{R}^{-1}$**; if $\mathbf{R}$ is (nearly) singular, apply a nugget $\mathbf{R}+\epsilon\mathbf{I}$ ($\epsilon\approx10^{-6}$) and show it helps. Then fit Kriging on the *full* training set with a GP library (e.g. `GaussianProcessRegressor` with `normalize_y=True` and an anisotropic `RBF(length_scale=np.ones(6))` kernel, or `fitrgp` with `'BasisFunction','constant'`); report test RMSE/$R^2$ and the predicted std at the test points nearest to and farthest from the training set, and comment on how the uncertainty relates to distance from the training data.

In [ ]:
# --- Part A: your own ordinary-Kriging predictor (small subsample) -----------------

# TODO: implement the correlation matrix R(x_i, x_j) = exp(-theta * ||x_i - x_j||^2)
# hint: def corr_matrix(X1, X2, theta):
#           d2 = ((X1[:, None, :] - X2[None, :, :]) ** 2).sum(-1)
#           return np.exp(-theta * d2)

# TODO: implement kriging_fit(X, y, theta, nugget=0.0) -> model dict
# hint: R = corr_matrix(X, X, theta) + nugget * np.eye(len(X))
# hint: Rinv = np.linalg.inv(R)
# hint: beta = (ones.T @ Rinv @ y) / (ones.T @ Rinv @ ones)   # ordinary Kriging mean

# TODO: implement kriging_predict(Xnew, model) -> yhat
# hint: r = corr_matrix(Xnew, X, theta)          # shape (n_new, p)
# hint: yhat = beta + r @ Rinv @ (y - beta)

# TODO: draw p=25 training points with a fixed seed
# hint: rng = np.random.default_rng(RNG)
# hint: sub_idx = rng.choice(len(Xtr_s), size=25, replace=False)

# TODO: for >= 3 theta values spanning orders of magnitude, report kappa(R) (nugget=0)
# hint: np.linalg.cond(corr_matrix(Xsub, Xsub, theta))

# TODO: for the worst-conditioned theta, show that a nugget (R + 1e-6*I) helps

# --- Part B: full-scale fit with a GP library ---------------------------------------

# TODO: fit GaussianProcessRegressor on the FULL scaled training set
# hint: kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBFKernel(length_scale=np.ones(6),
#                                                              length_scale_bounds=(1e-2, 1e3))
# hint: gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True,
#                                      n_restarts_optimizer=3, random_state=RNG)
# hint: yhat_gp, std_gp = gp.predict(Xte_s, return_std=True)

# TODO: report test RMSE/R^2

# TODO: find the nearest and farthest TEST point from the TRAINING set (in scaled 6-D space)
# hint: D = cdist(Xte_s, Xtr_s); min_dist = D.min(axis=1)
# hint: i_near, i_far = np.argmin(min_dist), np.argmax(min_dist)

# TODO: report std_gp at i_near and i_far, and comment on how uncertainty relates to distance

## Bonus (optional, no penalty for skipping) --- Comparing surrogate families

Collect your tuned models (CV-selected; Kriging by maximum likelihood) from questions 3--6 -- quadratic+ridge, RBF, ANN, Kriging -- into one table of test RMSE/$R^2$ for `CL`, each family's complexity knob, and whether it gives native uncertainty quantification (mirroring lecture's "Family / Complexity knob / Native UQ?" comparison). State which family you'd recommend and why, citing a property from the table.

In [ ]:
# TODO: build a summary table with one row per family
# hint: rows = [("Polynomial (quad+ridge)", f"lambda=...", rmse_..., r2_..., "No"),
#               ("RBF",                     f"spread=...", rmse_..., r2_..., "No"),
#               ("ANN",                     f"width=...",  rmse_..., r2_..., "No"),
#               ("Kriging/GP",              "theta (MLE)", rmse_..., r2_..., "Yes (std)")]
# hint: pd.DataFrame(rows, columns=["Family","Knob","Test RMSE","Test R2","Native UQ"])

# TODO: state which family you'd recommend and why (in the Answer cell below)